In [14]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
from helpers import config
import pandas as pd
import ast

In [2]:
# dem = ee.Image("USGS/SRTMGL1_003")
geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("geoReg")

frtc_lc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal")

# Waterbody 1
# Glacier 2
# Snow 3
# Forest 4
# Riverbed 5
# Built-up area 6
# Cropland 7
# Bare soil 8
# Bare rock 9
# Grassland 10
# Other wooded land 11

noncrop_ic = frtc_lc_ic.map(lambda image: image.neq(7).And(image.neq(3)).rename("noncrop"))
stable_noncrop_mask = noncrop_ic.sum().eq(22).unmask(0)

In [3]:
stable_noncrop_mask_eroded = stable_noncrop_mask.focalMin(radius=4, units='pixels') #Removing ag field boundary by 2 pixel
patch_size = stable_noncrop_mask_eroded.connectedPixelCount(maxSize=10, eightConnected=True)
stable_noncrop_mask_final = stable_noncrop_mask_eroded.updateMask(patch_size.gte(10))

In [ ]:
# geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("geoReg")
# lc_2022 = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal").filter(ee.Filter.eq("system:index", "lc2022")).first()
# strata = lc_2022.multiply(100).add(geo_region).rename("strata")

# geo_stable_noncrop = stable_noncrop_mask_final.addBands(strata)

# area_stable_noncrop_final_mask = geo_stable_noncrop.reduceRegion(
#     reducer = ee.Reducer.sum().unweighted().group(groupField = 1, groupName = "strata"),
#     geometry = config.ROI,
#     scale = 30,
#     maxPixels = 474870364
# )
# client = area_stable_noncrop_final_mask.getInfo()
# pd.DataFrame(client["groups"]).to_csv("outputs/test/geoReg_LC_nonCrop.csv", index=False)

In [10]:
geemap.ee_export_image_to_drive(
    image=stable_noncrop_mask_final.selfMask(),
    description="stable_noncrop_mask",
    scale=30,
    maxPixels= 992227573612
)

In [8]:
Map.addLayer(stable_noncrop_mask.selfMask(), {}, "noncrop")
Map.addLayer(stable_noncrop_mask_final.selfMask(), {}, "noncrop_final")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
lc2022 = frtc_lc_ic.filter(ee.Filter.eq("system:index","lc2022")).first()

In [43]:
lc2022.updateMask(stable_noncrop_mask_final).reduceRegion(
    geometry=config.ROI,
    scale=30,
    reducer=ee.Reducer.frequencyHistogram().unweighted(),
    maxPixels=207435182
)

In [44]:
stable_noncrop_mask.reduceRegion(
    geometry=config.ROI,
    scale=30,
    reducer=ee.Reducer.sum().unweighted(),
    maxPixels=207435182
)

In [4]:
stable_noncrop_points = stable_noncrop_mask_final.selfMask().rename('noncrop').stratifiedSample(
    numPoints=200000,     
    classBand='noncrop', 
    region=config.ROI, 
    scale=30, 
    geometries=True,
    dropNulls=True,
    tileScale = 4
)

In [5]:
def apply_spatial_thinning(points, distance_meters):
    """
    Removes spatial autocorrelation by ensuring no two points 
    are within the specified distance of each other.
    """
    # 1. Add a random value to each point
    points_with_random = points.randomColumn('random')

    # 2. Define a spatial filter for the 1,000-meter radius
    dist_filter = ee.Filter.withinDistance(
        distance=distance_meters,
        leftField='.geo',
        rightField='.geo',
        maxError=10
    )

    # 3. Create a join to find all neighbors within that distance
    join = ee.Join.saveAll(
        matchesKey='neighbors',
        measureKey='distance'
    )

    # 4. Apply the join to the FeatureCollection
    joined_points = join.apply(points_with_random, points_with_random, dist_filter)

    # 5. Function to evaluate each neighborhood
    def check_if_max(feature):
        # Get the list of all points within 1,000m (including itself)
        neighbors = ee.List(feature.get('neighbors'))
        
        # Extract the random values of all these neighbors
        neighbor_randoms = neighbors.map(lambda f: ee.Feature(f).get('random'))
        
        # Find the maximum random value in this cluster
        max_random = neighbor_randoms.reduce(ee.Reducer.max())
        
        # If THIS point's random value is the maximum, mark it to be kept
        is_max = ee.Number(feature.get('random')).eq(max_random)
        
        return feature.set('keep', is_max)

    # 6. Apply the evaluation and filter out the losers
    thinned_points = joined_points.map(check_if_max).filter(ee.Filter.eq('keep', 1))

    # 7. Clean up the temporary properties we added so your data stays clean
    def cleanup(f):
        return f.set('keep', None).set('neighbors', None).set('random', None)
        
    return thinned_points.map(cleanup)

In [6]:
stable_noncrop_points_filtered = apply_spatial_thinning(stable_noncrop_points, 2000)

In [7]:
geemap.ee_export_vector_to_asset(
    collection=stable_noncrop_points_filtered,
    description = "stable_noncrop_points_filtered",
    assetId = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered"
)

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered
Exporting stable_noncrop_points_filtered... Please check the Task Manager from the JavaScript Code Editor.


## Extract lc at each sample

In [6]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap

In [14]:
geometry = ee.FeatureCollection(
        [ee.Feature(
            ee.Geometry.Point([85.33987783757082, 27.696937701624826]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "0"
            }),
        ee.Feature(
            ee.Geometry.Point([85.3081204828345, 27.703777134221017]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "1"
            }),
        ee.Feature(
            ee.Geometry.Point([85.35103582707278, 27.730522797403868]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "2"
            }),
        ee.Feature(
            ee.Geometry.Point([85.43240331974856, 27.674136497886323]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "3"
            }),
        ee.Feature(
            ee.Geometry.Point([85.28271459904543, 27.694353804430413]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "4"
            }),
        ee.Feature(
            ee.Geometry.Point([85.32048010197512, 27.65802077690818]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "5"
            }),
        ee.Feature(
            ee.Geometry.Point([85.31231360594019, 27.73603718875524]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "6"
            }),
        ee.Feature(
            ee.Geometry.Point([85.3210683361648, 27.704277589831218]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "7"
            }),
        ee.Feature(
            ee.Geometry.Point([85.33514456907496, 27.726312744018685]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "8"
            }),
        ee.Feature(
            ee.Geometry.Point([85.4271504482308, 27.6711756489846]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "9"
            }),
        ee.Feature(
            ee.Geometry.Point([81.62638444051447, 28.06574187894065]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "10"
            }),
        ee.Feature(
            ee.Geometry.Point([84.87936769281445, 27.012286387463497]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "11"
            }),
        ee.Feature(
            ee.Geometry.Point([84.8711279467207, 27.012209920091703]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "12"
            }),
        ee.Feature(
            ee.Geometry.Point([83.46585124843101, 27.703384452706114]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "13"
            }),
        ee.Feature(
            ee.Geometry.Point([83.47855419032554, 27.68727304701869]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "14"
            }),
        ee.Feature(
            ee.Geometry.Point([84.42485675640842, 27.698581963092487]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "15"
            }),
        ee.Feature(
            ee.Geometry.Point([84.42253932781955, 27.692958193254093]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "16"
            }),
        ee.Feature(
            ee.Geometry.Point([87.28108689050356, 26.45700258064935]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "17"
            }),
        ee.Feature(
            ee.Geometry.Point([87.27602287988344, 26.456464688536116]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "18"
            }),
        ee.Feature(
            ee.Geometry.Point([83.9849716882535, 28.208941373222356]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "19"
            }),
        ee.Feature(
            ee.Geometry.Point([83.98711745546541, 28.24047715749985]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "20"
            }),
        ee.Feature(
            ee.Geometry.Point([83.98282592104158, 28.232008029881186]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "21"
            }),
        ee.Feature(
            ee.Geometry.Point([83.98986403749666, 28.22361385625498]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "22"
            }),
        ee.Feature(
            ee.Geometry.Point([85.30145180149081, 27.684113761632762]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "23"
            }),
        ee.Feature(
            ee.Geometry.Point([85.30376923007968, 27.730239050292617]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "24"
            }),
        ee.Feature(
            ee.Geometry.Point([85.35372269077304, 27.716563074623448]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "25"
            }),
        ee.Feature(
            ee.Geometry.Point([85.34548294467929, 27.69133399828552]),
            {
              "first": 6,
              "noncrop": 1,
              "system:index": "26"
            })])
    
geometry2 = ee.FeatureCollection(
        [ee.Feature(
            ee.Geometry.Point([82.07799755671093, 29.526193411461797]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "0"
            }),
        ee.Feature(
            ee.Geometry.Point([82.10048519709179, 29.53351216877182]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "1"
            }),
        ee.Feature(
            ee.Geometry.Point([83.8456503170634, 28.70049839101554]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "2"
            }),
        ee.Feature(
            ee.Geometry.Point([83.85766661345012, 28.680621078954534]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "3"
            }),
        ee.Feature(
            ee.Geometry.Point([81.78148939810099, 30.128283565191218]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "4"
            }),
        ee.Feature(
            ee.Geometry.Point([81.68432905874552, 30.140160496129866]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "5"
            }),
        ee.Feature(
            ee.Geometry.Point([82.94106626492126, 29.180313804205955]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "6"
            }),
        ee.Feature(
            ee.Geometry.Point([82.94518613796814, 29.21268186238499]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "7"
            }),
        ee.Feature(
            ee.Geometry.Point([82.95273923855407, 29.20189364504071]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "8"
            }),
        ee.Feature(
            ee.Geometry.Point([86.7874743842543, 27.92391023628197]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "9"
            }),
        ee.Feature(
            ee.Geometry.Point([83.93982196350967, 28.22235502627141]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "10"
            }),
        ee.Feature(
            ee.Geometry.Point([83.95080829163467, 28.21123727093072]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "11"
            }),
        ee.Feature(
            ee.Geometry.Point([84.09491801758682, 28.18143295381903]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "12"
            }),
        ee.Feature(
            ee.Geometry.Point([84.09878039856827, 28.17212685831053]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "13"
            }),
        ee.Feature(
            ee.Geometry.Point([84.10847926636612, 28.17023527647973]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "14"
            }),
        ee.Feature(
            ee.Geometry.Point([84.10942340393936, 28.147155285686736]),
            {
              "first": 1,
              "noncrop": 1,
              "system:index": "15"
            })])


In [16]:
filtered_points = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered")
roi = ee.FeatureCollection("projects/ee-joshisur231/assets/pa_effectiveness/nepal_boundary"),
filtered_points = filtered_points.merge(geometry).merge(geometry2)
frtc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal")
lc= frtc_ic.filter(ee.Filter.eq("system:index", "lc2022")).first()
  
points_lc = lc.reduceRegions(collection= filtered_points, reducer= ee.Reducer.first(), scale=30).map(lambda feat : feat.set("lc2022", ee.Number(feat.get("first")).toInt())).select(propertySelectors=["noncrop", "lc2022"], retainGeometry= True)

In [17]:
points_lc.aggregate_histogram("lc2022")

In [20]:
geemap.ee_export_vector_to_asset(
    collection= points_lc,
    description = "nonCriop",
    assetId = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_withLC"
)

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_withLC
Exporting nonCriop... Please check the Task Manager from the JavaScript Code Editor.


## Extract NDVI

In [5]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config

In [6]:
start_date = "2000-01-01"
end_date = "2022-12-31"

points = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_noncrop_points_filtered_withLC")

In [7]:
l8_ndvi_col = ee.ImageCollection("LANDSAT/COMPOSITES/C02/T1_L2_8DAY_NDVI")\
    .filterBounds(config.ROI)\
    .filterDate(start_date, end_date)
    
def extract_point_values(image):
    l_with_time_band = image.addBands(image.metadata("system:time_start").rename("time"))
    points_with_ndvi = l_with_time_band.reduceRegions(collection = points, scale=30, reducer=ee.Reducer.first())
    return points_with_ndvi

points_with_ndvi = l8_ndvi_col.map(extract_point_values).flatten()

In [8]:
geemap.ee_export_vector_to_drive(
    collection=points_with_ndvi,
    description="stable_nonCrop_withNDVI",
    fileFormat = "CSV"
)

Exporting stable_nonCrop_withNDVI... Please check the Task Manager from the JavaScript Code Editor.
